# 40 · Production RAG：全景图 + 总复习

> 到此，从“什么是 LLM”到“可观测上线”的全部知识都到齐了。最后这一课把它们**串成一张全景图**，并给出一条从 0 到生产的**行动路线**。

**本文件覆盖知识点**：Production RAG / 十模块总复习 / 端到端串联 / 上线检查清单 / 常见坑 / 演进路径

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 一张全景图看完整 RAG

```text
 知识源 ──加载/解析/清洗──> 切分 ──> Embedding ──> 向量索引
 (PDF/网页/表)     04-06       07-09      10-12       13/14
                                        │
 用户问题 ──> 查询理解 ──> 混合检索 ──> 重排 ──> 上下文工程 ──> 生成
            18-21      15-17      22       23-26        01-03
              │
              └─ 全程被 评估(33/34)、安全(35)、缓存(36)、观测(37) 包围
```

十**大模块复习**（按主干流动）：

| # | 模块 | 关键 Notebook | 一句话 |
|---|------|--------------|--------|
| 1 | **文档加载 / 解析 / 清洗** | 04-06 | 源头烂则全链烂；别把脏数据焊进 chunk |
| 2 | **切分 Chunking / 元数据** | 07-09 | 粒度与语义完整性是检索精度之母 |
| 3 | **Embedding** | 10-12 | 文本进向量空间，专业领域要挑模型 |
| 4 | **向量库与索引** | 13/14 | 海量下 ANN/HNSW 换取速度 |
| 5 | **检索策略** | 15-17 | 稠密+稀疏混合，RRF 融合最稳 |
| 6 | **查询理解与改写** | 18-21 | 先改写问题再检索，收益立竿见影 |
| 7 | **重排 Rerank** | 22 | 用精排模型榨干 top-k 质量 |
| 8 | **上下文工程与提示词** | 23-26 | 引证、位置、防注入全在提示词里 |
| 9 | **高级范式** | 27-32 | Graph/Agentic/Self-RAG/多模态/SQL/代码 |
| 10 | **评估与工程化** | 33-40 | 量化指标 + 安全 + 可观测 = 可上线 |


## 2. 从 0 到生产的行动路线

```text
第1步 能用   手搓最小链路: 读档→切分→embedding→检索→LLM
第2步 能准   加混合检索 + 重排；建 50~100 题评测集量化
第3步 能稳   上下文工程/引证/提示词加固；Graph/Agent 按需
第4步 能上线 缓存提速 + 安全过滤 + 日志链路 + 灰度回滚
```

### 上线检查清单
- [ ] 评测集存在，Recall/NDCG/Faithfulness 有基线数值
- [ ] 检索片段带元数据（来源/页码），回答带引证
- [ ] 防注入隔离 + PII 脱敏 + 行级权限过滤
- [ ] 缓存命中策略、超时与降级
- [ ] 每次问答落 trace_id 日志，指标接入监控
- [ ] 提示词/索引/代码版本化，可回滚

In [ ]:
# 知识点·真调说明：上线检查清单 —— 让模型按场景现场生成一份可勾选的 RAG 上线清单
_llm_live(
    prompt='你负责把星云客服问答 RAG 从 demo 推向生产（私有知识库 + 混合检索 + qwen-plus 生成）。'
           '请产出一份“上线前检查清单”：按 数据/检索/生成/工程/安全/监控 六类分组，每类 2~3 条可勾选事项，'
           '每条是“动作 + 为什么”，不要空话。',
    system='你是资深 SRE/LLM 应用交付负责人，直接输出清单，条条可执行、可勾选。',
    fallback='未配置 Key 的固定样例：\n'
             '【数据】□ 评测集≥50 题并记录 Recall/NDCG/Faithfulness 基线（无基线就无法判断上线是否回退）\n'
             '【检索】□ 混合检索 + RRF，片段带来源/页码元数据（专有名词召回差多半是只上了稠密）\n'
             '【生成】□ 引证 + 限长截断 + 低温度；无引用时明说“未检索到依据”（压制幻觉）\n'
             '【工程】□ 提示词/索引版本化、可回滚；LLM 超时熔断与优雅降级（上游挂了不能全挂）\n'
             '【安全】□ 防注入隔离 + PII 脱敏 + 行级权限过滤（知识库越权是最隐蔽的生产事故）\n'
             '【监控】□ 每次问答落 trace_id 日志 + P95/空检索率/负面反馈上监控（可回溯、可告警）',
    temperature=0.2,
)
print('→ 40 课的终点就是把这张清单一条条打勾——每个勾都对应前面某课的“工程化”动作。')

## 3. 高频翻车点（避开即胜利）

| 坑 | 症状 | 对策 |
|----|------|------|
| 不做切分/乱切 | 检索答非所问 | 语义完整切分 + 重叠 |
| 只上稠密检索 | 专有名词/编号检索差 | 混合检索 + RRF |
| 不做评测就调参 | “感觉好了”不可复现 | 固定评测集量化 |
| 上下文一股脑塞 | 贵、慢、Lost-in-the-Middle | 重排截断 + 位置安排 |
| 不溯源 | 幻觉难排查 | 引证 + 元数据 + 日志 |
| 没有降级 | 上游一挂全挂 | 优雅降级 + 缓存 |

In [ ]:
# 知识点·真调说明：演进路径 —— 让模型对比“渐进加模块”与“一步到位上复杂范式”两条路线
_llm_live(
    prompt='现有 RAG 已上线但效果平庸（答对率一般、偶有幻觉）。团队想“升级”，两派意见：\n'
           '路线 A：渐进——加混合检索/重排、建评测集量化、逐项修上下文与引证；\n'
           '路线 B：一步到位——直接引入 GraphRAG / Agentic / 长上下文全塞。\n'
           '请站在“要可上线、可回退”的立场：①两条路线各自的成本与风险 ②你会怎么走 '
           '③什么信号出现才值得上路线 B 的组件。分 3 点回答，每点不超过 3 句话。',
    system='你是 RAG 交付架构师，观点务实：默认增量，用评测数据驱动要不要上高级组件。',
    fallback='未配置 Key 的固定样例：\n'
             '① A 成本低、每步可回退、收益可量化但偏“增量”；B 架构与排查复杂度高，'
             '数据/评测不到位时“看起来炫但不稳”。\n'
             '② 先走 A：把评测集与混合检索/重排基线立住，先在“检索与上下文工程”解决大部分问题，'
             '这是性价比最高的一段。\n'
             '③ 当出现“需多跳/关系型问答、单一知识源不够”且评测显示检索是瓶颈时再上 GraphRAG/Agentic；'
             '当需要“极长原文兜底”时才补长上下文——都以评测数字先降为触发信号。',
    temperature=0.2,
)
print('→ 演进不是“越复杂越好”，而是“每个高级技巧对应一个评测数字的改善”——结语那句话的现场版。')

In [ ]:
# 最后一课：一条“最小可验证 RAG”全流程(纯本地, 不依赖API)，作为复习闭环
from dotenv import load_dotenv; load_dotenv()
import os, numpy as np

# 语料
docs = ['星云支持公有云与私有化两种部署',
        '报销需登录OA提交发票走审批',
        'RAG检索增强生成可减少幻觉']
def embed(s):
    v = np.zeros(32); [np.add.at(v, ord(c) % 32, 1) for c in s]; return v / (np.linalg.norm(v)+1e-9)
E = np.stack([embed(d) for d in docs])

def retrieve(q, k=1):                       # 1 检索
    sim = E @ embed(q)
    return [docs[i] for i in np.argsort(-sim)[:k]]

q = '星云可以部署在客户自己的机房吗？'
ctx = retrieve(q)

# 2 生成
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
if API_KEY and '你的' not in API_KEY:
    from dashscope import Generation
    p = f'只依据资料回答: {ctx}\n问题: {q}'
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':p}],
                        api_key=API_KEY, result_format='message')
    print('答案:', r.output.choices[0].message.content)
else:
    print(f'[演示] 命中资料: {ctx[0]}')
    print('[演示] 配置 DASHSCOPE_API_KEY 后此段将调用 qwen-plus 生成最终答案。')
    print('\n恭喜通关 40 课！剩下的就是把这条链路接上真数据、真评测、真监控。')

## 结语

40 节课是一张地图，不是终点：

> **先跑通最小链路，再用量化评测驱动优化；每一个“高级技巧”都该对应一个评测数字的改善。**

祝你的 RAG 从 demo 走向生产 🚀